## tl;dr

Real locally trained XGBoost_depth3; experimental Kentucky susceptibility, **not NER forecasting**. Held-out ROC-AUC 0.844, average precision 0.090. Only 4/22 events detected at the selection-set threshold. Share with caveats; not suitable for operational safety decisions.

## Context & Methods

Source: [NASA EIS case study](https://git.smce.nasa.gov/eis-freshwater/landslides), commit `1cfada85265c38e39b8531a2a5d06ef288ec9285`. 2010–2014 common window; event/background point-date grain. 15 candidate configurations; fixed 0.25-degree geographic blocks; separate training, calibration, selection and test partitions. No test-driven retuning.

### Key Assumptions

Background points are not verified negatives. Same-day rain makes this retrospective classification, not a forecast. No storm-group or temporal holdout, spatial buffer, or NER data. Native geology encodings are not universal categories. Source license is unspecified: raw data and artifacts remain local. This notebook checks existing results; it does not retrain.

In [1]:
from pathlib import Path
import sys, json, csv, hashlib
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/python').is_dir()) / 'backend/python'
sys.path.insert(0, str(root))
from training.nasa_experiment import prepare, split_data, HASHES
from app.experimental import ExperimentalModel
run = root / 'runs/nasa-kentucky-v1'
report = json.loads((run / 'report.json').read_text())

## Data

Verify source hashes, counts and geographic separation; no network is needed.

In [2]:
x, y, groups, records, audit = prepare(root / 'data/nasa-kentucky')
assert audit['sha256'] == HASHES
assert audit['usable_records'] == report['data_quality']['usable_records']
parts = split_data(x, y, groups)
saved = json.loads((run / 'split-membership.json').read_text())
assert {k: [records[i]['id'] for i in v] for k, v in parts.items()} == saved
for a in parts:
    for b in parts:
        if a != b:
            assert not set(groups[parts[a]]) & set(groups[parts[b]])
print(json.dumps({k: audit[k] for k in ['raw_counts', 'dropped', 'usable_records', 'positive_records', 'background_records', 'spatial_blocks']}, indent=2))

{
  "raw_counts": {
    "landslides.csv": 323,
    "random.csv": 19447
  },
  "dropped": {
    "outside_common_2010_2014_window": 188,
    "missing_or_invalid": 2,
    "duplicate_features_or_location_date": 6
  },
  "usable_records": 19574,
  "positive_records": 138,
  "background_records": 19436,
  "spatial_blocks": 208
}


## Results

Recompute metrics directly from saved predictions and reproduce predictions from the saved classifier.

In [3]:
with (run / 'test-predictions.csv').open() as stream:
    rows = list(csv.DictReader(stream))
labels = np.array([int(r['label']) for r in rows])
predictions = np.array([float(r['score']) for r in rows])
model = ExperimentalModel()
np.testing.assert_allclose(model.model.predict_proba(x[parts['test']])[:, 1], predictions, rtol=1e-7)
assert [r['sample_id'] for r in rows] == saved['test']
assert np.array_equal(labels, y[parts['test']])
auc = roc_auc_score(labels, predictions)
ap = average_precision_score(labels, predictions)
assert abs(auc-report['test']['roc_auc']) < 1e-12
assert abs(ap-report['test']['average_precision']) < 1e-12
cm = confusion_matrix(labels, predictions >= report['threshold']).tolist()
assert cm == report['test']['confusion_matrix']
print(json.dumps({'ROC_AUC': auc, 'average_precision': ap, 'background_AP_baseline': float(labels.mean()), 'confusion_matrix_TN_FP_FN_TP': cm, 'threshold_chosen_on_selection': report['threshold']}, indent=2))

{
  "ROC_AUC": 0.8440241381237079,
  "average_precision": 0.09001671913492212,
  "background_AP_baseline": 0.006715506715506716,
  "confusion_matrix_TN_FP_FN_TP": [
    [
      3239,
      15
    ],
    [
      18,
      4
    ]
  ],
  "threshold_chosen_on_selection": 0.06962264150943397
}


## Takeaways

The software uses a trained classifier, not hard-coded route winners. The 25 synthetic files test software only and are never training data. Low event recall, few held-out events, assumed negatives and geographic transfer prevent a safety/NER accuracy claim. Flood, weather, access, delays and route scoring remain rules or supplied values. Next scientific step: labelled NER events plus matched non-event periods, before-event weather, local geology/soil inputs and storm/location/time-separated external validation.